# London Flip Finder

**End-to-end valuation and mispricing detection for the London residential market (2008–2016).**

This notebook is the narrative layer. The pipeline it drives lives in `src/lff/`, one module per
stage, still organised as a sequence of small pure functions — each takes data in and returns data
out, so any cell can be re-run without corrupting another cell's state, and the whole run is
deterministic given `CONFIG.seed`. The notebook carries the argument and the evidence; the package
carries the machinery; `tests/` carries the checks that used to run only at the end of a full
run.

| Stage | Sections | What happens |
|---|---|---|
| Setup | 1–3 | Imports, one `CONFIG` object, self-bootstrapping data download |
| Data | 4–7 | Load, clean, spatially join, merge into one master table |
| Analysis | 8 | Exploratory data analysis |
| Features | 9–10 | Temporal/market engineering, leakage-safe four-way chronological split |
| Models | 11–13 | Ridge → XGBoost/CatBoost (plain and detrended) → two Mixture-of-Experts designs |
| Decision | 14–16 | Test evaluation, target-transform and feature-group experiments, conformal safety bound, flip scanner |
| Assurance | 17–18 | Automated self-checks, persisted artifacts, known limitations |

**The business question.** Given a property's physical, spatial, temporal and macroeconomic
context, what is it worth — and can we identify listings priced below a *statistically
guaranteed* floor, so that a buyer has a quantified margin of safety?

**What the finished pipeline found** (full run, 59,946 transactions):

| | |
|---|---|
| Selected on validation (§14) | `3-seed average detrended (XGB)` — **13.30 %** test MdAPE |
| Best single model | `XGBoost detrended-market (capped)` — 13.12 % val → 13.48 % test |
| Ridge (baseline) | 16.72 % test — beaten by 3.4 pp |
| Conformal floor | prediction × **0.7484**, **88.96 %** empirical coverage vs a 90 % target |
| Flip candidates | 978 of 8,859 test properties (11.04 %), median margin £76,166 |
| Crime ablation (§14.5) | removing crime costs **+0.01 pp** validation MdAPE — far below the 0.15 pp bar, so crime does **not** justify the narrow window |

## Contents

- [1. Imports and environment](#1-imports-and-environment)
- [2. Configuration](#2-configuration)
- [3. Data acquisition](#3-data-acquisition)
- [4. Loading the raw sources](#4-loading-the-raw-sources)
- [5. Cleaning and per-source feature construction](#5-cleaning-and-per-source-feature-construction)
- [6. Spatial engineering](#6-spatial-engineering)
- [7. Building the master table](#7-building-the-master-table)
- [8. Exploratory data analysis](#8-exploratory-data-analysis)
  - [8.1 Spatial, safety and macroeconomic drivers](#81-spatial-safety-and-macroeconomic-drivers)
  - [8.2 Does crime price into London property?](#82-does-crime-price-into-london-property)
- [9. Temporal and market feature engineering](#9-temporal-and-market-feature-engineering)
- [10. Chronological partitioning and encoding](#10-chronological-partitioning-and-encoding)
- [11. Metrics](#11-metrics)
- [12. Models](#12-models)
  - [12.1 Detrending the target: a fix, not just a diagnosis](#121-detrending-the-target-a-fix-not-just-a-diagnosis)
  - [12.2 The same diagnosis, reached without the test set](#122-the-same-diagnosis-reached-without-the-test-set)
- [13. Validation leaderboard](#13-validation-leaderboard)
- [14. Held-out test evaluation](#14-held-out-test-evaluation)
  - [14.1 Repeat-property diagnostic](#141-repeat-property-diagnostic)
  - [14.2 Error diagnostics and feature importance](#142-error-diagnostics-and-feature-importance)
  - [14.3 Choosing a target transform: three options, tested head to head](#143-choosing-a-target-transform-three-options-tested-head-to-head)
  - [14.4 Is the Mixture of Experts needed, once the base model is fixed?](#144-is-the-mixture-of-experts-needed-once-the-base-model-is-fixed)
  - [14.5 Feature-group ablation: is crime worth what it costs?](#145-feature-group-ablation-is-crime-worth-what-it-costs)
  - [14.6 What a property's own history is worth](#146-what-a-propertys-own-history-is-worth)
- [15. Conformal safety bound and the flip scanner](#15-conformal-safety-bound-and-the-flip-scanner)
- [16. Persisting the run](#16-persisting-the-run)
- [17. Automated self-checks](#17-automated-self-checks)
- [18. Limitations and where to take this next](#18-limitations-and-where-to-take-this-next)

---
## Project overview

**What this notebook does.** It builds a model of what a London residential property is worth,
using only what a buyer standing at the transaction date could actually have known — physical
attributes, location, the state of the market and the neighbourhood, and the cost of borrowing at
the time. It then wraps that prediction in a calibrated lower bound, so a listing can be flagged
as a **flip candidate**: not "the model thinks this looks cheap," but "the price sits below a
floor that holds roughly 90% of the time, with a quantified margin of safety."

**Datasets used.**

| Dataset | Grain | What it contributes |
|---|---|---|
| Land-Registry-derived price history | one row per sale | the target price and the property's physical attributes |
| Met Police crime by LSOA | LSOA × category × month | a neighbourhood-safety signal |
| Bank of England base rate | one row per rate change | the cost of borrowing at the time of sale |
| TfL station geodata (Feb 2022 snapshot, filtered) | one row per station | transport connectivity |
| GLA borough boundaries | polygon per borough | administrative and spatial context |

**Models used, and why.** Ridge regression is the baseline — linear, fast, and hard to overfit, so
it doubles as a sanity check on everything that follows. XGBoost and CatBoost are the workhorses:
gradient-boosted trees pick up non-linear interactions a linear model can't, and CatBoost's native
categorical handling avoids having to hand-encode `propertyType`, `tenure`, `borough` and
`outcode`. On top of those sit two Mixture-of-Experts designs, which route a sale to one of
several specialised sub-models — by price tier, or by which base model tends to be more accurate
— to test whether segmenting the market beats a single model or a plain ensemble average (§12
covers how that test turns out). One structural weakness of plain gradient boosting is worth
flagging here too: a tree's prediction is a leaf constant, so it can't extrapolate past the
highest price level it saw in training. §12.1 works around this by predicting price *relative to
the market level* rather than price outright, which is what lets the tree models beat Ridge at
all.

**Why a confidence interval, not just a point prediction.** A single predicted price says nothing
about how much to trust it, and "this looks underpriced" isn't something you can act on without
knowing how often it's wrong. §15 applies **conformal prediction** to turn the point estimate into
a calibrated lower bound instead — a price floor with a target coverage rate (90% here) that is
checked empirically rather than assumed. A property is only flagged as a flip candidate if its
price sits below that floor, which turns "the model likes this one" into a quantified,
falsifiable margin of safety rather than a hunch.

---
## Related work

None of the ingredients here are new on their own — hedonic valuation, repeat-sales history, and
distribution-free prediction intervals each have a literature. What is specific to this project is
the combination: a hedonic model of London stock in 2008–2016, given the property's own sale
history, wrapped in a *one-sided* calibrated floor so the output is a screen a buyer can act on
rather than a point estimate.

| Work | What it is | Where it touches this project |
|---|---|---|
| Rosen (1974), [*Hedonic Prices and Implicit Markets*](https://www.journals.uchicago.edu/doi/10.1086/260169), JPE 82(1):34–55 | The theory that prices a differentiated good as a bundle of measured attributes, each carrying an implicit price. | The framing of §5–§9. Every feature — floor area, tenure, distance to a station, borough — is an attribute whose implicit price the model is estimating; the tree models just drop the linear-in-attributes assumption. |
| Bailey, Muth & Nourse (1963), [*A Regression Method for Real Estate Price Index Construction*](https://www.semanticscholar.org/paper/A-Regression-Method-for-Real-Estate-Price-Index-Bailey-Muth/8384788b906b9cbde02c20fede181f7163fc29eb), JASA 58:933–942; extended by [Case & Shiller (1987)](https://www.nber.org/system/files/working_papers/w2506/w2506.pdf) | Repeat sales: use properties sold more than once to separate market movement from property quality, since the property is held fixed between the two sales. | Both halves of that idea are used here, in opposite directions. §12.1 divides out the market level to get a stationary target; §14.6 goes the other way and feeds the *previous sale price* back in as a feature — the single strongest signal in the data (+0.9 pp MdAPE). §14.1 also reports errors on repeat properties separately. |
| Gibbons (2004), [*The Costs of Urban Property Crime*](https://onlinelibrary.wiley.com/doi/abs/10.1111/j.1468-0297.2004.00254.x), Economic Journal 114(499):F441–F463 | A hedonic study of London specifically: criminal damage capitalises into prices (≈1% per tenth of a standard deviation in Inner London), burglary does not. | The closest prior work to §8.2 and §14.5, and it predicts what we find — a real but small effect that is category-dependent. Worth reading as the reason the crime block earns so little once location is already in the model, rather than as a contradiction of it. |
| Lei, G'Sell, Rinaldo, Tibshirani & Wasserman (2018), [*Distribution-Free Predictive Inference for Regression*](https://arxiv.org/abs/1604.04173), JASA 113(523):1094–1111 | The reference treatment of split conformal prediction: finite-sample marginal coverage on top of any regressor, with no distributional assumptions. | §15 is split conformal with a *ratio* nonconformity score (actual/predicted) and only the lower tail kept, because a flip screen cares about the floor and not the ceiling. |
| Romano, Patterson & Candès (2019), [*Conformalized Quantile Regression*](https://papers.nips.cc/paper/8613-conformalized-quantile-regression), NeurIPS 32:3538–3548 | Conformal intervals that adapt their width to the input, rather than one global correction. | The obvious next step for §15. The multiplier q here is a single constant across the whole market, so coverage holds on average but the floor is loose for easy properties and tight for hard ones. [MAPIE](https://github.com/scikit-learn-contrib/MAPIE) is the usual scikit-learn implementation of both this and the split method above. |
| [Zillow Prize](https://www.zillow.com/z/info/zillow-prize/) (Kaggle, 2017–2019) | The largest public competition on automated valuation: 3,800+ teams predicting the Zestimate's log error; the winners improved on the benchmark by ~13%, and Zillow reports the national median error falling from ~4.5% to under 4%. | The practitioner reference point for §12–§14 — gradient-boosted ensembles over property, location and time features are what won there too. The error rates are *not* comparable to the 13.30% MdAPE here: a different market, no listing or interior data, and every transaction scored rather than on-market homes only. |


---
## 1. Imports and environment

The pipeline lives in `src/lff/`; this cell imports it. No cell below re-imports anything, and no
cell installs packages at runtime — dependencies come from `requirements.txt` (see `README.md`).

| Module | Stage |
|---|---|
| `lff.config` | paths, tunables, seeding (§2) |
| `lff.ingest` | dataset download and raw reads (§3–4) |
| `lff.clean` | per-source cleaning (§5) |
| `lff.spatial` | projection, nearest-neighbour and point-in-polygon joins (§6) |
| `lff.master` | the joined master table (§7) |
| `lff.features` | temporal, market, and the feature registry (§9) |
| `lff.split` | chronological partitioning, encoding, training variants (§10) |
| `lff.metrics` | one metric implementation, one results registry (§11) |
| `lff.models` | trainers, deflators, mixture-of-experts (§12) |
| `lff.analysis` | diagnostics and design studies (§12.2, §14.1–14.5) |
| `lff.conformal` | conformal bound and flip scanner (§15) |
| `lff.plots` | every figure |
| `lff.persist` | run artifacts and self-checks (§16–17) |

Warnings are filtered *narrowly* rather than with a blanket `filterwarnings('ignore')`, so genuine
problems still surface; `apply_notebook_theme()` installs those two filters along with the chart
theme and frame display width.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Importable from a clone with no install step.
_SRC = Path.cwd() / "src"
if _SRC.exists() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import geopandas as gpd
import numpy as np
import pandas as pd
import xgboost as xgb

import lff
from lff.analysis import (
    ablation_study,
    check_moe_necessity,
    crime_resolution_study,
    extrapolation_bias,
    prior_sale_study,
    repeat_property_diagnostic,
    summarise_target_transform,
)
from lff.clean import build_crime_features, build_rate_curve, clean_houses
from lff.config import Config, set_seeds
from lff.conformal import calibrate_conformal, scan_for_flips
from lff.crime import build_crime_features_lsoa
from lff.external import fetch_lsoa_boundaries
from lff.features.market import add_market_features
from lff.features.prior_sale import (
    add_prior_sale_features,
    assert_no_lookahead,
    build_sale_history,
)
from lff.features.registry import (
    CATEGORICAL_FEATURES,
    FEATURE_GROUPS,
    FEATURES,
    NUMERIC_FEATURES,
    TARGET,
)
from lff.features.temporal import add_temporal_features
from lff.ingest import ensure_dataset, load_raw
from lff.maps import (
    plot_crime_and_price_maps,
    plot_crime_change,
    plot_crime_within_borough,
)
from lff.master import build_master_table
from lff.metrics import ResultsRegistry, evaluate
from lff.models import train_all
from lff.notebook import apply_notebook_theme
from lff.persist import persist_run, run_self_checks
from lff.plots import (
    plot_ablation,
    plot_crime_and_market,
    plot_error_diagnostics,
    plot_feature_importance,
    plot_flip_margins,
    plot_leaderboard,
    plot_price_clip_comparison,
    plot_price_per_sqm_by_borough,
    plot_price_vs_area,
    plot_property_characteristics,
    plot_tube_premium,
)
from lff.spatial import split_station_networks
from lff.split import chronological_split, training_variants

apply_notebook_theme()

import catboost  # noqa: E402  -- imported here purely to report its version
import sklearn  # noqa: E402

print(f"python      {sys.version.split()[0]}")
for _mod in (np, pd, gpd, xgb):
    print(f"{_mod.__name__:<12}{_mod.__version__}")
print(f"{'sklearn':<12}{sklearn.__version__}")
print(f"{'catboost':<12}{catboost.__version__}")
print(f"{'lff':<12}{lff.__version__}  (src/lff -- see README for the module map)")

---
## 2. Configuration

A single frozen `Config` object is the only source of truth for paths, filters, split ratios and
hyper-parameters. Nothing below hardcodes a threshold — change a value here and re-run.

`FAST_MODE` shrinks every model's iteration budget so the whole notebook executes in a couple of
minutes for a smoke test; set `LFF_FAST_MODE=1` in the environment to enable it.

**Data location** is resolved in this order: `$LFF_DATA_DIR` → `./data/data_for_ds_project`.
Section 3 downloads the data if it is not there.

In [ ]:
CONFIG = Config()
CONFIG.artifact_dir.mkdir(parents=True, exist_ok=True)

set_seeds(CONFIG.seed)

print(f"data_dir     {CONFIG.data_dir}")
print(f"artifact_dir {CONFIG.artifact_dir}")
print(f"fast_mode    {CONFIG.fast_mode}")
print(f"split        {CONFIG.train_frac:.0%} train / {CONFIG.val_frac:.0%} val / "
      f"{CONFIG.calib_frac:.0%} calib / {CONFIG.test_frac:.0%} test")

# The bar a feature group must clear to justify the data it costs. Defined here rather than in
# section 14.5 because section 8.2 consumes it too, and the two must use the same number.
ABLATION_GATE_PP = 0.15

---
## 3. Data acquisition

The datasets total ~1.8 GB, which is far past GitHub's file limit, so they ship as a release
asset rather than living in the repository. This cell makes the notebook self-bootstrapping: if
the files are missing it downloads and extracts them, and if they are already present it does
nothing. That is what makes the notebook runnable on a fresh machine without any manual setup
steps.

In [ ]:
ensure_dataset(CONFIG)

---
## 4. Loading the raw sources

Four sources feed the model:

| Source | Grain | Contributes |
|---|---|---|
| Land-Registry-derived price history | one row per sale event | target price + physical attributes |
| Met Police crime by LSOA | LSOA × category × month | neighbourhood safety |
| Bank of England base rate | one row per rate change | cost of borrowing |
| TfL station geodata | one row per station | transport connectivity |

One deliberate exclusion:

* **The `saleEstimate_*` and `rentEstimate_*` columns are not read.** They are a third party's
  *model output* for the same property, so using them to predict price would be target leakage
  dressed up as a feature.


In [ ]:
RAW = load_raw(CONFIG)

# Two extra sources for section 8.2. Boundaries come from ONS and are cached under
# data/external/; the crime file is re-read at its native LSOA grain, which load_raw
# deliberately does not do -- it reads only the four columns the borough aggregation needs.
LSOA_BOUNDS = fetch_lsoa_boundaries(CONFIG)
CRIME_LSOA_RAW = pd.read_csv(
    CONFIG.crime_csv,
    usecols=["lsoa_code", "major_category", "value", "year", "month"],
    dtype={"lsoa_code": "category", "major_category": "category",
           "value": "int16", "year": "int16", "month": "int8"},
)

---
## 5. Cleaning and per-source feature construction

Three independent, pure transformations — each takes one raw frame and returns a tidy one.

**Crime is engineered with two lags, never with contemporaneous data.** `crime_volume` is the
month immediately *before* the sale (a short-term safety signal), and `crime_volume_prev_12m` is
the rolling twelve-month sum computed with `closed='left'` so the current month is excluded (a
stable "reputation" signal). Both are strictly backward-looking, which is what a buyer standing
at the transaction date could actually have known.

**Interest rates** arrive as a sparse list of change dates, forward-filled onto a daily calendar so
every sale is matched to the rate in force on the day it completed. 

In [ ]:
HOUSES = clean_houses(RAW["houses"], CONFIG)
CRIME = build_crime_features(RAW["crime"])
RATES = build_rate_curve(RAW["boe"], CONFIG)
LSOA_CRIME = build_crime_features_lsoa(CRIME_LSOA_RAW, LSOA_BOUNDS)

---
## 6. Spatial engineering

Location is the single biggest driver of price in this market, so getting the spatial work right
— and fast — matters more here than almost anywhere else in the pipeline.

**Projection comes first.** Before any distance gets computed, everything is reprojected to the
British National Grid (EPSG:27700). Latitude and longitude aren't great units for measuring
distance: at London's latitude, a degree of longitude only covers about 62% of the ground a degree
of latitude does, so computing distance directly in degrees quietly distorts geography along the
east-west axis. BNG is metric instead, so a distance of `1000` just means 1,000 metres, in any
direction.

**Finding the nearest station uses a k-d tree (cKDTree).** Checking every one of the ~80,000
properties against all ~400 stations by brute force means 32 million distance calculations — slow
for no real benefit. A k-d tree answers each nearest-neighbour query in `O(log n)`, so the whole
join finishes in well under a second. As a bonus, the same lookup also returns the neighbour's
index, which hands over the station's fare zone for free.

### The station file needed cleaning up first

`Stations_20220221.csv` is a snapshot of the TfL network from February 2022, but every transaction
in this dataset happened before 2017. Used as-is, it would credit properties with transport links
that simply didn't exist yet at the time of sale:

| Group | Count | Treatment |
|---|---:|---|
| London Underground | 270 | kept — includes the 6 that later also gained Elizabeth Line service |
| London Overground | 113 | kept in the wider transit measure |
| DLR | 45 | kept in the wider transit measure |
| Elizabeth-Line-only | 33 | **excluded** — the line didn't open until May 2022 |
| Tramlink-only | 39 | **excluded** — Croydon trams aren't heavy rail |

Filtering on the Underground/Overground/DLR flags handles both exclusions in one pass. It also
avoids a subtler mistake: a handful of stations (Paddington among them) were already
long-standing Underground stops that only *also* gained Elizabeth Line service in 2022.
Excluding anything tagged Elizabeth Line outright would wrongly erase a transport link that
genuinely existed at the time of sale, rather than just dropping the part of the station that
didn't exist yet.

Two distinct features are derived rather than one, as each addresses a separate question:
`distance_to_underground_m` measures proximity specifically to the London Underground network,
while `distance_to_transit_m` measures proximity to any rail-based transit mode. An earlier,
undifferentiated measure, `distance_to_nearest_tube_m`, conflated these two questions: computed
across all 471 stations without regard to service type, it treated a Croydon Tramlink stop as
equivalent to an Underground station.

*A limitation remains unaddressed:* several Overground and DLR line extensions opened during
the 2008–2016 study period, so the transit-distance measure is likely overstated for properties
sold in the earlier years of the window. A fully accurate treatment would require station-level
opening dates, which are not available in the present dataset. The feature is retained
nonetheless: the affected extensions are a small minority of the network, the resulting bias is
one-directional and partial rather than a fabricated link (a property near a pre-existing line
is merely credited with slightly earlier or better service than it had), and the alternative —
dropping transit connectivity from the feature set entirely — would discard a materially
stronger signal than the one it is meant to correct for.

In [ ]:
# The station snapshot is filtered here so section 6's narrative has visible output; the
# same split runs inside build_master_table.
UNDERGROUND, HEAVY_RAIL = split_station_networks(RAW["stations"])

---
## 7. Building the master table

One orchestration function chains sections 4–6 into the modelling table. It is deliberately the
only place that writes `df_master`, and it takes no globals other than `CONFIG`, so re-running it
always produces the same result.

Row filters applied here, and why:

* **`price_per_sqm >= 1500`** — removes symbolic transfers (£1 family transactions, parking
  spaces, lease extensions) that are legally sales but economically meaningless. Note this is a
  *ratio* filter, so genuinely expensive homes survive as long as their price-to-size ratio is
  plausible.
* **Deduplication** on date + geometry + size + price — the same completion occasionally appears
  more than once in the history file.
* **Chronological sort** — mandatory before any rolling window or time-based split.

The result is cached to Parquet so later runs skip the ~1 GB crime read.

In [ ]:
df_master = build_master_table(CONFIG, HOUSES, CRIME, RATES, RAW["stations"],
                               lsoa_crime=LSOA_CRIME, lsoa_boundaries=LSOA_BOUNDS)
df_master.head(3)

---
## 8. Exploratory data analysis

Each figure is a function of a DataFrame, called once, and nothing here mutates `df_master` or
rebinds a source frame. That discipline is what keeps the notebook re-runnable top to bottom: an
EDA cell that reassigns a name like `df_tube` destroys the raw station table for every cell below
it, and does so silently.

**Chart conventions used throughout.** Colours come from a validated categorical palette applied
in fixed slot order, so a colour always means the same series. Single-series charts get one flat
hue and no legend (the title names the series); the correlation matrix gets a diverging blue↔red
ramp with a neutral grey midpoint, because its data has a meaningful zero. Grid and axes are kept
recessive so the marks carry the message.

Visualisations clip at the 95th percentile of price. Without it, a handful of £20 M sales
would stretch every axis to fit them, and the £200k–£800k range where most of the market
actually sits would collapse into an unreadable sliver against the left edge of every chart.
That is a *display* choice only, made purely so the charts stay legible — the models below see
the untruncated data, so nothing about what gets learned is affected. The figure directly
below shows both versions side by side, full and clipped, so the effect is visible rather
than just asserted.

In [ ]:
plot_price_clip_comparison(df_master, CONFIG)

**The two panels make the same point the paragraph above made in words.** In the full-data panel, the bulk of London's sales sit bunched hard against the left edge, because a small number of multi-million-pound properties stretch the axis out far enough to flatten everything else into a spike. Clipped to the 95th percentile, the same data resolves into a proper right-skewed distribution with a visible peak and shoulder. Every other chart in this section uses the clipped view for exactly this reason -- the shape underneath is identical, only how visible it is changes.

In [ ]:
plot_property_characteristics(df_master, CONFIG)

**Size and location dominate; everything else is secondary.** Price is heavily right-skewed even after clipping, with a sharp peak around £250,000-£300,000 and a long tail beyond it. It rises with room count, from a median around £320,000 at two rooms to roughly £950,000 at eight, though the boxes widen and overlap heavily at the top end -- room count alone does not pin down price for larger properties. Detached properties top the median-price ranking by property type, roughly double the cheapest terraced categories.

The correlation panel puts numbers on the physical drivers: `floorAreaSqM` (r = 0.57) and `bathrooms` (r = 0.52) are price's strongest positive correlates, `distance_to_center_m` (r = -0.36) and `distance_to_underground_m` (r = -0.30) its strongest negative ones. `floorAreaSqM` and `total_rooms` are themselves correlated at r = 0.85 -- a reminder that several of these nominally independent drivers are really measuring the same thing, size, from different angles.

### 8.1 Spatial, safety and macroeconomic drivers

Three external forces, each shown on its own scale.

The price-versus-interest-rate figure deliberately uses **two stacked panels sharing one x-axis
rather than a single chart with twin y-axes**. A dual-axis chart lets whoever draws it decide
where the two lines appear to cross by choosing the scales, which manufactures a visual
correlation that may not exist in the data. Stacked panels show the same co-movement without that
degree of freedom.

In [ ]:
plot_tube_premium(df_master, CONFIG)
plot_crime_and_market(df_master, CONFIG)
plot_price_vs_area(df_master, CONFIG)
plot_price_per_sqm_by_borough(df_master, CONFIG)

**Four charts, four confirmations of what a London buyer already prices in.** Mean price falls with distance from the nearest station -- close to £900,000-£950,000 within 500m, under £350,000 beyond 3km -- though the very first band (0-250m) is not the cheapest, likely a shorter walk traded against being right next to the station itself. Price tracks floor area closely on a log-log scale, with the spread widening at the top end, where finish and location start to matter as much as square metreage. By borough, median price per square metre spans roughly 5x, from over £11,000 in Kensington and Chelsea to around £2,300 in Bexley -- inner versus outer London in one number.

The crime-band boxplot is the one result worth flagging rather than nodding past: median price climbs, broadly, from the `Low` band through to `Severe`, topping out over £470,000 against roughly £390,000 in `Low`. Read naively, that says more crime means higher prices, backwards from what buyers actually report caring about -- and it is exactly the confound §8.2 exists to unpick, at LSOA rather than borough grain. The price-and-rate panel above it shows the base rate collapsing from 5.5% to 0.5% during 2008-2009 while price dipped and then climbed steadily for years afterward, consistent with cheaper borrowing supporting higher prices over the medium term.

### 8.2 Does crime price into London property?

Section 14.5 removes crime from the feature set, measures the cost at **+0.01 pp** of validation
MdAPE, and concludes that crime does not earn the 2008–2016 window it forces on the whole
project — a window that discards 86 % of the available sale records. That is a large decision to
rest on one number, and the number was measured under conditions that stack against crime three
separate ways:

* **Resolution.** The raw file is `london_crime_by_lsoa.csv` and carries 4,835 LSOA codes.
  `build_crime_features` sums them to 33 boroughs, a 147× loss. A borough mean averages
  Hampstead with Kilburn, and any relationship between local safety and local price is
  overwhelmingly a *within*-borough effect — exactly the variation a borough mean removes.
* **Count, not rate.** `crime_volume` is a raw count, so a populous borough scores high
  mechanically. LSOAs are built to hold roughly 1,500 residents each, so an LSOA count is
  already close to a per-capita rate.
* **One series.** Burglary and drug offences are summed together, which assumes a buyer prices
  them identically.

All three are fixed before the verdict is taken, here and in 14.5. The point is not to rescue
crime. It is that *"crime does not matter"* and *"crime measured 147× too coarsely, as an
unnormalised count, does not matter"* are different claims, and only the second had been
tested.

In [ ]:
# Left: what a place costs. Right: how much crime it records. One hue each, so the two
# panels cannot be misread as sharing a scale.
lsoa_gdf = plot_crime_and_price_maps(df_master, LSOA_CRIME, LSOA_BOUNDS)

Both maps run darkest in the middle, and that is the problem with reading them directly.
Central London is simultaneously the most expensive and the most crime-recording part of the
city, so the raw association between crime and price comes out **positive** — naively, more
crime looks like more money. The crime-quartile boxplot in 8.1 has this confound baked into it.
(2,562 of the 2,863 LSOAs in Greater London record at least five sales in the window and are
used for this comparison.)

Two ways to strip centrality out. Subtracting each borough's own mean from both variables leaves
the within-borough contrast: two streets in one borough, one safer than the other. Differencing
over time removes every fixed feature of a place at once — its architecture, its parks, its
distance to the centre, its reputation — and asks a sharper question: did the LSOAs where crime
fell see faster price growth than the ones where it rose?

In [ ]:
crime_partial = plot_crime_within_borough(lsoa_gdf)

In [ ]:
crime_differenced = plot_crime_change(df_master, LSOA_CRIME)

**The association survives neither treatment.** Within-borough, the correlation drops from
r = +0.30 across boroughs to r = +0.00 within them — the entire apparent relationship is which
borough a property sits in, not its crime level. Differenced over time, the same verdict comes
from an unrelated direction: r = -0.01 between the change in local crime and the change in local
price, across 863 LSOAs. Whatever the raw scatter measures, it is not crime.

That is the exploratory answer, and it is not the whole answer: a feature can carry no marginal
correlation and still earn its place by interacting with others inside a model. Section 14.5 asks
that question in the model's own terms, and lands on the same verdict by an independent route —
removing crime costs only +0.01 pp of validation MdAPE.

---
## 9. Temporal and market feature engineering

This is where most of the leakage risk in a property model lives, so every feature here is built
to answer one question: *would a buyer standing at the transaction date have known this?*

| Feature | Construction | Why it cannot leak |
|---|---|---|
| `days_since_start` | days elapsed since the first transaction | derived from the row's own date |
| `month_sin`, `month_cos` | cyclic encoding of calendar month | December and January end up adjacent, as they are in reality |
| `market_median_rolling_3m` / `_12m` | market-wide monthly median, `.shift(1)` then rolled | the shift drops the current month before the window opens |
| `lagged_borough_median_sqm` | borough £/sqm median, stamped onto the *following* month | a month's own price never informs its own prediction |
| `avg_room_size` | floor area ÷ total rooms | a within-row ratio |

`avg_room_size` needs care: `total_rooms` can be zero, which yields ±∞ rather than NaN. Those are
converted to NaN explicitly and then **left as NaN**. Filling them with the column median here
would leak: this function runs on the whole table, before the split, so that median would be
computed over validation, calibration and test rows and then baked into a training feature. It is a
small leak but a real one. Leaving NaN defers the
imputation to each model, where it happens inside a pipeline fitted on the training split alone.

In [ ]:
# A property's own transaction history, read from the WHOLE 1995-2024 file rather than the
# modelling window: a 2009 sale's previous sale is usually pre-2008, and clipping first would
# discard most of the signal. Section 14.6 measures what it is worth.
SALE_HISTORY = build_sale_history(RAW["houses"])

df_model = add_market_features(add_temporal_features(df_master))
df_model = add_prior_sale_features(df_model, SALE_HISTORY)
df_model = df_model.dropna(subset=[TARGET, "date"]).reset_index(drop=True)
assert_no_lookahead(df_model)

print(f"Modelling table: {df_model.shape}")
print(f"Features: {len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical")
missing = df_model[FEATURES].isna().mean().sort_values(ascending=False)
print("\nMissing-value rate by feature (top 6):")
print(missing.head(6).to_string())

---
## 10. Chronological partitioning and encoding

**The split is by time, not at random.** A random split would let the model learn from June 2016
to predict January 2016, which is not a situation that ever occurs in production. Sorting by date
and cutting **60 / 15 / 10 / 15** into train / validation / calibration / test reproduces the real
task: train on the past, forecast the future.

**Why calibration gets its own slice.** A single validation set cannot serve three roles at once —
early stopping, model selection, and calibrating the conformal safety bound — without the "90 %
guarantee" in section 15 being partly calibrated on data the model was already tuned against. That
is circular, so calibration gets a dedicated slice that nothing else touches. The ordering of the
two middle slices was tested both ways; validation-then-calibration is kept because it produces
smaller validation-to-test drift for the tree models, at the cost of a little exchangeability on
the calibration split.

### One evaluation universe, three training variants

Comparing a "raw", a "capped" and a "cleaned" model is only meaningful if all three are scored on
the same rows; letting the evaluation set vary alongside the training set compares models and test
sets at once. Here the evaluation universe is fixed (validation and test rows under the £4 M cap,
the standard market the product actually targets) and only the **training** data varies:

| Variant | Training rows |
|---|---|
| `raw` | every transaction, including £4 M+ |
| `capped` | transactions at or below the cap |
| `cleaned` | capped, minus multivariate anomalies |

**Anomalies are removed from training only.** Fitting `IsolationForest` on the whole dataset and
dropping the flagged rows everywhere would delete the hard cases from validation and test as well —
marking your own exam after removing the difficult questions. A production model does not get to
refuse the awkward listings. Here the forest is fitted on the training slice alone and filters only
that.

### Encoding

Two representations, both fitted on training data only:

* **Native categoricals** (`category` dtype) for XGBoost, with the level set pinned from the
  training split so a category means the same integer code everywhere. CatBoost gets the raw
  strings via `cat_features`. These columns must never be coerced with
  `pd.to_numeric(..., errors='coerce')`, which turns all four **entirely NaN** and leaves the
  CatBoost models and every Mixture-of-Experts router training on dead columns.
* **Smoothed target encoding** for the models that require numeric input (the routers). The
  smoothing pulls low-frequency categories toward the global mean so a borough with three sales
  does not get a confident price estimate.

In [ ]:
SPLITS = chronological_split(df_model, CONFIG)

# One fixed evaluation universe for every model: the standard market, under the cap.
VAL_EVAL = SPLITS.val[SPLITS.val[TARGET] <= CONFIG.price_cap]
CALIB_EVAL = SPLITS.calib[SPLITS.calib[TARGET] <= CONFIG.price_cap]
TEST_EVAL = SPLITS.test[SPLITS.test[TARGET] <= CONFIG.price_cap]
print(f"\nEvaluation universe (price <= \N{POUND SIGN}{CONFIG.price_cap:,.0f}): "
      f"{len(VAL_EVAL):,} validation, {len(CALIB_EVAL):,} calibration, "
      f"{len(TEST_EVAL):,} test rows")

In [ ]:
VARIANTS = training_variants(SPLITS, CONFIG)

---
## 11. Metrics

One metric function, used by every model. Near-identical variants defined across several cells are
how a leaderboard ends up silently mixing two different spellings of the same metric, and with them
two different numbers.

**MdAPE (median absolute percentage error) is the headline metric.** MAE is reported in pounds and
is easy to explain, but it is dominated by the expensive tail: a 10 % miss on a £3 M house
contributes thirty times more than a 10 % miss on a £100 k flat, even though both are equally
wrong in the only sense the buyer cares about. The median of the percentage errors is robust to
that tail and answers the practical question — *what does a typical valuation get wrong by?*

In [ ]:
RESULTS = ResultsRegistry()

---
## 12. Models

Every trainer returns a `ModelBundle` exposing the same `predict(X)` signature. That uniformity is
not cosmetic — it is what lets the leaderboard, the test evaluation and the conformal calibration
below all run through one code path. It also closes off a whole class of error: a conformal step
that reaches for a concrete estimator such as `expert_0.predict(X_test)` computes its guarantee
from whatever that name happens to point at, which need not be the model under evaluation, and
fails silently when it is not.

**All targets are modelled in log space** (`log1p` in, `expm1` out). Property prices are strongly
right-skewed; training on the log makes the loss approximately proportional rather than absolute,
which is the same reason MdAPE is the headline metric.

| Model | Role |
|---|---|
| Ridge | linear baseline — how far do straight-line relationships get you? |
| XGBoost | the workhorse gradient-boosted tree |
| CatBoost | alternative booster with native categorical handling |
| XGBoost / CatBoost detrended | same trees, target is `log(price / market level)` instead of `log(price)` — see 12.1 |
| Luxury MoE | router splits standard from luxury, one expert each, soft-weighted |
| Error-driven MoE | three diverse experts, router learns which wins per property |
| 3-seed average | **the honest control** for the error-driven MoE (see below) |

The 3-seed average is the control that makes the MoE result interpretable. The error-driven MoE
trains three experts that differ only by random seed, then routes between them.
Averaging those same three experts costs nothing extra and is what the routing must beat to
justify its complexity — without that control, a "Mixture of Experts" that merely beats a single
model has proved only that ensembling works.

**Both MoE architectures are trained on the detrended recipe as well as the plain one.** Judging an
MoE only on plain `log(price)` -- the target that costs the single trees their accuracy -- would
test routing against a handicapped baseline and credit it for a fix it did not make. Training both
makes the comparison the informative one: does routing or averaging add anything *on top of* the
best single model, rather than on top of a target that is already mis-levelled. See 14.4.

### 12.1 Detrending the target: a fix, not just a diagnosis

Gradient-boosted trees are normally the stronger choice on tabular data of this shape, so a plain
Ridge regression beating them on test by five points of MdAPE or more reads as a symptom rather
than a verdict on the architecture. Detrending — predicting the ratio to the market level instead
of the raw price — addresses the mechanism behind that symptom rather than the symptom itself.

The plain XGBoost/CatBoost models above train on `log1p(price)`. A tree's prediction is a
constant per leaf, so once a test-time value of a trending feature (`days_since_start`,
`market_median_rolling_3m`) exceeds anything seen in training, every such row lands in the same
boundary leaf and the model flat-lines at the last price level it learned -- it cannot
extrapolate a rising market. Ridge suffers less, because it multiplies by a coefficient instead of
splitting and so at least projects the trend forward -- though section 12.2 shows it is not immune
either, only less exposed.

That is a testable hypothesis, not a guess: the plain trees under-predict by a large, positive
mean residual — a one-directional level error, not noise. Section 12.2 measures that bias on the
**validation** set, which is the version of the evidence this notebook is entitled to act on. The
same effect measures +£128,318 on test; section 12.2 exists so that the conclusion does not rest on
the held-out set, and section 18 records the order in which the two were actually observed.

**The fix removes the need to extrapolate at all.** Instead of predicting price, predict the
*ratio* of price to the lagged whole-market median (`market_median_rolling_3m`, already a
leakage-safe feature computed in section 9):

$$y = \log\!\left(\frac{\text{price}}{\text{market level}}\right)$$

"How much is this property worth relative to where the market already is" is close to
stationary across time, even while the market itself trends -- so a tree only has to
*interpolate* within the range of ratios it saw in training, never extrapolate beyond it. The
market level is multiplied back onto the prediction at inference time.

The second change is the training objective: XGBoost defaults to squared error with no
`objective` set, which does not match MdAPE (a median of relative errors). Setting
`objective="reg:absoluteerror"` on log-ratio space optimises something close to the actual
metric. CatBoost already trains with an MAE loss, so only the detrending applies there --
isolating which of the two changes is doing the work.

These are real competitors, not a side experiment: they are registered in `train_all()` like
every other model, so whichever one wins on validation becomes `BEST` and flows through test
evaluation, the repeat-property diagnostic and conformal calibration exactly like any other
model would.

One caveat worth stating plainly rather than leaving implied: selecting *among* the candidates on
validation is clean, but the detrended family entered the candidate pool at all because of an
observation first made on test. Competing fairly afterwards does not retroactively make the
candidate pool test-blind. Section 12.2 is the repair — it shows the same diagnosis is reachable
from validation — and section 18 records the history.

**Two deflators, compared head to head rather than assumed.** `market_median_rolling_3m` is one
number per calendar month, shared by every property regardless of size or borough -- it removes
the market-wide time trend and nothing else. A finer alternative is `lagged_borough_median_sqm x
floorAreaSqM`: a size- and location-scaled deflator, personalised per row. It is *not* built by
bucketing on a coarse category like bedroom count, deliberately -- slicing a 3-month window by
bedrooms x borough x property type would leave many cells with only a handful of sales, and a
noisy deflator injects noise directly into every label it divides. Floor area is continuous, so
`lagged_borough_median_sqm x floorAreaSqM` scales smoothly to the exact property instead of
falling into a sparse bucket, while borough alone (33 categories, thousands of sales each) stays
statistically stable month to month. Both deflators train an XGBoost and a CatBoost model each,
so this doubles as a test of whether finer-grained detrending is worth its added complexity.

In [ ]:
BUNDLES = train_all(SPLITS, VARIANTS, CONFIG, VAL_EVAL, RESULTS)

### 12.2 The same diagnosis, reached without the test set

Section 12.1's argument rests on a specific claim: the plain trees fail because they cannot
extrapolate a rising market, and the signature of that failure is a **one-directional level error**
— under-prediction that grows as the market moves past anything seen in training — rather than
symmetric noise.

That claim was originally spotted on the *test* set, which is an honest description of how this
project actually unfolded and is disclosed as such in section 18. But a diagnosis that can only be
made on held-out data is a diagnosis you were not entitled to make. So the cell below re-derives it
on **validation** alone.

If the bias is equally visible there, the detrending fix was reachable from data the notebook is
allowed to look at, and the test set merely confirmed something already decidable. The mechanism is
structural — a leaf constant cannot exceed the largest value it was fitted on, whichever future
window you point it at — so the effect should appear in any period after training, not just the
last one.

**Read the Ridge row carefully — it complicates the story rather than confirming it.** Section 12.1
frames this as "trees cannot extrapolate, Ridge can". The measurement below is more nuanced: Ridge
*also* under-predicts, because the market rose faster than any linear projection of its training
window. What separates the models is the **size** of that level error, not its presence. The plain
trees carry the largest bias, Ridge sits in between, and the detrended trees are the least biased
of the three. So the accurate claim is that a non-stationary target biases *every* model here and
punishes the trees hardest — and that detrending, not architecture, is what removes most of it.

In [ ]:
_bias_models = ["XGBoost (capped)", "XGBoost detrended-market (capped)", "Ridge (baseline)"]
bias_val = extrapolation_bias(_bias_models, VAL_EVAL, "val", SPLITS, BUNDLES)
print("Signed-residual diagnostic, computed on VALIDATION only:\n")
print(bias_val.to_string(index=False, formatters={
    "Mean residual (val)": "\N{POUND SIGN}{:,.0f}".format,
    "Median residual": "\N{POUND SIGN}{:,.0f}".format,
    "Under-predicted %": "{:.1f}%".format}))

_plain = bias_val[bias_val["Model"] == "XGBoost (capped)"]
_fixed = bias_val[bias_val["Model"] == "XGBoost detrended-market (capped)"]
if not _plain.empty and not _fixed.empty:
    plain_bias = float(_plain.iloc[0]["Mean residual (val)"])
    fixed_bias = float(_fixed.iloc[0]["Mean residual (val)"])
    print(f"\nPlain XGBoost under-predicts validation by a mean of \N{POUND SIGN}{plain_bias:,.0f}; "
          f"detrending moves that to \N{POUND SIGN}{fixed_bias:,.0f}.")
    verdict = ("visible on validation alone" if plain_bias > 0 and abs(fixed_bias) < abs(plain_bias)
               else "NOT reproduced on validation -- treat section 12.1's reasoning with caution")
    print(f"The extrapolation signature is {verdict}, so the fix did not require the test set.")

bias_val

---
## 13. Validation leaderboard

The table below is generated from the results registry — every row was appended by `evaluate()`
at training time, so it is impossible for the leaderboard to disagree with what the models actually
scored. A hand-assembled table, with metric dictionaries typed into a `DataFrame` literal, carries
no such guarantee: that is how a £4 M cap ends up labelled "<£5M".

Read `Trained on` and `MdAPE` together: every row is scored on the *same* validation rows, so
differences reflect the model and its training data, nothing else.

In [ ]:
val_board = RESULTS.frame("val")
plot_leaderboard(val_board, "Validation performance, identical evaluation rows")
val_board.style.format({
    "R2": "{:.3f}", "MAE": "\N{POUND SIGN}{:,.0f}", "RMSE": "\N{POUND SIGN}{:,.0f}",
    "MdAPE": "{:.2f}%", "MAPE": "{:.2f}%", "within_25pct": "{:.1f}%",
}).background_gradient(cmap="Blues_r", subset=["MdAPE", "MAE"])

---
## 14. Held-out test evaluation

**This is the section that matters.**

Every number in section 13 comes from the validation set — the same data used for early stopping
and for choosing between architectures. Reporting those figures as the model's accuracy is
circular: they measure how well the winner fits the set it won on. Nothing before this cell has
read the test split.

The procedure: pick the winner by **validation** MdAPE, then score on test. The gap between the two
is the honest measure of how much of the validation performance was selection effect.

**How often test is read, precisely.** Every model is scored once here so the drift column exists
and the leaderboard is comparable; the repeat-property diagnostic (14.1) and the error diagnostics
(14.2) then read it for the selected model, and section 15 reads it once more to report conformal
coverage. That is reporting, not selection: the winner was fixed by validation before this cell
ran, and every *decision* in sections 14.3, 14.4 and 14.5 is made on validation. Section 18 records
the two points during this project's development where that protocol was not followed.

**Why this order.** 14.1 and 14.2 diagnose the model just revealed above. 14.3 and 14.4 resolve
the two open design questions from section 12 -- which target transform, and whether the
Mixture-of-Experts architecture earned its complexity. 14.5 comes last because it isn't really a
statement about *this* model at all: it's a scoping question about next iteration's data, and
putting it before the model-design verdicts would break the "questions about what we built,
then questions about what to build next" arc.

In [ ]:
BEST_NAME = val_board.iloc[0]["Model"]
BEST = BUNDLES[BEST_NAME]
print(f"Selected on validation MdAPE: {BEST_NAME}\n")

print("Scoring every model once on the held-out test set:")
for bundle in BUNDLES.values():
    evaluate(bundle, TEST_EVAL, "test", RESULTS, SPLITS)

test_board = RESULTS.frame("test")
comparison = (
    val_board[["Model", "MdAPE", "MAE"]]
    .merge(test_board[["Model", "MdAPE", "MAE"]], on="Model", suffixes=(" (val)", " (test)"))
)
comparison["MdAPE drift"] = comparison["MdAPE (test)"] - comparison["MdAPE (val)"]
comparison = comparison.sort_values("MdAPE (test)").reset_index(drop=True)
print()
comparison.style.format({
    "MdAPE (val)": "{:.2f}%", "MdAPE (test)": "{:.2f}%", "MdAPE drift": "{:+.2f}pp",
    "MAE (val)": "\N{POUND SIGN}{:,.0f}", "MAE (test)": "\N{POUND SIGN}{:,.0f}",
})

### 14.1 Repeat-property diagnostic

The source file is a price *history*: one row per sale event, so a dwelling that changed hands
three times between 2008 and 2016 contributes three rows. A chronological split cuts on **time**,
not on **property**, which means a flat sold in 2010 (train) and again in 2016 (test) appears on
both sides of the wall.

That is not automatically cheating — forecasting a known building's next sale price is a real
business task, and the lagged features are still strictly historical. But it does mean the
headline test metric blends two very different problems: re-valuing a property the model has
already seen, and valuing one it has never seen. The cell below separates them, because the second
number is the one that generalises to new stock.

In [ ]:
repeat_property_diagnostic(BEST, SPLITS, TEST_EVAL)

### 14.2 Error diagnostics and feature importance

Two questions about the selected model specifically, now that section 14.1 has ruled out
memorisation as the main story: where do its remaining errors live (biased against price level,
or just noisy?), and which inputs is it actually leaning on.

In [ ]:
plot_error_diagnostics(BEST, TEST_EVAL, SPLITS)

In [ ]:
plot_feature_importance(BEST)

### 14.3 Choosing a target transform: three options, tested head to head

Once a market-wide deflator addresses the extrapolation problem, the obvious follow-up is whether a
more granular one does better — deflating by borough and property size rather than by the whole
market at once. The intuition is that a deflator built from genuinely comparable properties should
carry more signal than one lumping the whole city together. That is a hypothesis, so it is tested
head to head rather than adopted.

Section 12.1 diagnosed why the plain trees underperform: trained on `log(price)`, a tree cannot
extrapolate a rising market, and it under-predicts the test set by a mean of **+£128,318**. That
diagnosis motivated a fix, and the fix itself came in two candidate forms -- tested side by side
rather than assumed to work:

1. **Regular** -- predict `log(price)` directly (the untransformed target, still trained above for
   comparison).
2. **Detrended, market-wide** -- predict `log(price / market_median_rolling_3m)`, one deflator
   shared by every property sold in the same calendar month.
3. **Detrended, borough-scaled** -- predict `log(price / (lagged_borough_median_sqm x
   floorAreaSqM))`, a deflator personalised to each property's own size and borough.

Option 3 is the one the intuition above favours. The comparison below reports what the data
does, which is not what that intuition predicts.

In [ ]:
transform_comparison = summarise_target_transform(RESULTS)
print(transform_comparison.to_string(index=False,
      formatters={"Validation MdAPE": "{:.2f}%".format, "Test MdAPE": "{:.2f}%".format,
                  "Test MAE": "£{:,.0f}".format}))

# The winner is chosen on VALIDATION. Picking it by test MdAPE would be using the held-out set to
# make a design decision -- the exact failure mode section 14.4 and the limitations section warn
# about. The test column above is reported so the reader can judge whether the choice generalised.
for backend in transform_comparison["Backend"].unique():
    sub = transform_comparison[transform_comparison["Backend"] == backend].set_index("Target transform")
    winner = sub["Validation MdAPE"].idxmin()
    line = (f"\n{backend}: '{winner}' wins on validation at "
            f"{sub.loc[winner, 'Validation MdAPE']:.2f}% MdAPE")
    if "Test MdAPE" in sub.columns:
        test_winner = sub["Test MdAPE"].idxmin()
        agreement = "and is also best on test" if test_winner == winner else \
                    f"but '{test_winner}' scored best on test"
        line += f" -- {agreement}"
    print(line)

transform_comparison

#### Why market-wide, not borough-scaled

**Both detrended options crush the regular target for both backends** -- confirming section
12.1's diagnosis was correct: the target's non-stationarity, not the tree architecture, was the
problem. Between the two detrended options, the *simpler, coarser* deflator wins for both
backends, and by a wide margin for CatBoost -- the opposite of what section 14.3's hypothesis
predicted.

**Why market-wide, not borough-scaled.** `lagged_borough_median_sqm` is estimated from far fewer
sales per month than the whole-market median -- one borough's transactions in a 3-month window,
against the entire city's. Because the deflator is a *divisor*, a noisier deflator injects that
noise directly into every training label it is divided into, not just into one more feature the
model can choose to weight down. The whole-market median, estimated from thousands of sales, does
not have that problem.

Crucially, using the coarser deflator does not cost the model any borough- or size-specific
signal: `borough`, `floorAreaSqM` and `lagged_borough_median_sqm` itself are still ordinary input
features (section 9), so the tree remains free to learn "this borough commands a premium" or
"larger properties are worth proportionally more" directly, from *uncorrupted* labels. Folding
that same information into the deflator does not add anything the tree could not already reach --
it only adds the deflator's own estimation noise to the target. The general lesson: a detrending
deflator should remove only what the model architecture genuinely cannot learn on its own (here,
the market-wide time trend); anything the model is already capable of learning from a feature
should stay a feature, not get folded into the label.

**Chosen going forward: the market-wide deflator.** It is simpler -- one already-computed column,
no dependency on floor area or borough being non-missing -- and it wins on validation for both
backends, which is the comparison the choice is made on. The test column above is printed so that
claim can be checked rather than taken on trust; whether the two agree is stated by the cell
itself rather than asserted here.

`XGBoost detrended-market (capped)` is the strongest *single* model on this recipe. It is not
necessarily what section 14 selects as `BEST` -- that is whichever model wins validation overall,
including the Mixture-of-Experts variants, which section 14.4 examines.

### 14.4 Is the Mixture of Experts needed, once the base model is fixed?

Both MoE designs sit on top of a base model, so a verdict reached while that base is mis-levelled
is confounded: routing can look unremarkable purely because the expert underneath it is
handicapped. Rebuilt on the detrended target, each component — the routing itself, the
luxury/standard split — can be asked separately whether it earns the complexity it adds.

Section 12.1 fixed the trees; section 14.3 picked the best deflator. This asks the remaining
question directly: given the best possible single model, does wrapping it in a Mixture of
Experts — luxury routing, error-driven routing, or even a plain multi-seed average — improve on
it at all, or does complexity added on top of an already-good model just add noise?

Three comparisons settle it, using the results already computed above:

1. **Luxury-routed MoE vs. the single detrended model.** Does splitting standard from luxury
   stock and blending two experts beat one model trained on everything?
2. **Error-routed MoE vs. the single detrended model.** Does a router that learns per-property
   which of three seed-diverse experts to trust beat one model?
3. **Error-routed MoE vs. its own 3-seed average control.** The check already used earlier in
   this notebook (section 12), now repeated on the fixed base: does the routing itself add
   anything beyond plain ensembling?

One nuance the table below has to handle: section 14 selects `BEST_NAME` from *every* trained
candidate, MoE included, and when an MoE variant narrowly wins that pool on validation -- as one
does here -- that model is what sections 15 (conformal) and 18 (persisted artifact) use, unchanged
from section 14's choice. This section does **not** override that selection with the comparison
below: doing so would mean using test-set results to decide which model gets deployed, which
defeats the point of holding test out at all. What this section *does* do is report that a
validation win this close (0.12pp) does not reliably predict a test win -- context for reading the
rest of the notebook, not a correction applied after the fact.

In [ ]:
# BEST_NAME (section 14) is whichever model wins on validation MdAPE, and that can legitimately
# be an MoE variant -- it is here, by 0.12pp on validation. Comparing an MoE against BEST_NAME
# in that case would compare it against itself. The question here is specifically "does wrapping
# the best single model in an MoE help", so the baseline must exclude MoE/average variants even
# if one of them is the overall validation winner.
single_models = val_board[~val_board["Model"].str.contains("MoE|average", case=False, regex=True)]
best_single_name = single_models.iloc[0]["Model"]
if best_single_name != BEST_NAME:
    print(f"Note: the overall validation winner ({BEST_NAME}) is itself an MoE variant. "
          f"Using the best single model ({best_single_name}) as the necessity baseline instead.\n")

moe_necessity = check_moe_necessity(RESULTS, best_single_name)
print(moe_necessity.to_string(index=False,
      formatters={"Test MdAPE": "{:.2f}%".format, "Test MAE": "£{:,.0f}".format,
                  "Test R2": "{:.3f}".format, "MdAPE delta vs. single model": "{:+.2f}pp".format}))

for _, row in moe_necessity.iterrows():
    if row["Variant"] == "Single model":
        continue
    verb = "beats" if row["MdAPE delta vs. single model"] < 0 else "loses to"
    print(f"\n{row['Variant']} {verb} the single model by "
          f"{abs(row['MdAPE delta vs. single model']):.2f}pp on test MdAPE")

# The routing-vs-averaging verdict is decided on VALIDATION. It is a statement about whether an
# architectural component earns its place, i.e. a decision -- and decisions do not read test.
# The test delta is printed beside it as disclosure only.
def _mdape(board: pd.DataFrame, model: str) -> float | None:
    match = board[board["Model"] == model]
    return float(match.iloc[0]["MdAPE"]) if not match.empty else None

ROUTING_GATE_PP = 0.05
_moe_name, _avg_name = "MoE - error routing detrended (XGB)", "3-seed average detrended (XGB)"
val_moe, val_avg = _mdape(val_board, _moe_name), _mdape(val_board, _avg_name)
test_moe, test_avg = _mdape(RESULTS.frame("test"), _moe_name), _mdape(RESULTS.frame("test"), _avg_name)

if val_moe is not None and val_avg is not None:
    routing_gain = val_avg - val_moe
    verdict = ("earns its complexity" if routing_gain > ROUTING_GATE_PP
               else "does not clearly beat plain averaging")
    print(f"\nRouting vs. its own averaging control, on validation: {routing_gain:+.3f}pp "
          f"-- the routing mechanism {verdict}.")
    if test_moe is not None and test_avg is not None:
        print(f"   (for disclosure, the same gap on test is {test_avg - test_moe:+.3f}pp -- "
              f"not used to reach the verdict above)")

# Sections 15 (conformal) and 18 (persisted artifact) still use BEST from section 14, chosen
# on validation MdAPE alone, before this comparison existed. That is deliberate: overriding the
# selection with the finding above would mean using test-set results to decide which model gets
# deployed, which defeats the point of holding test out in the first place. When BEST_NAME happens
# to be an MoE variant, as it is here, the honest response is to disclose that its validation edge
# is noisy near a tie (the table above), not to quietly substitute a different model after the
# fact. The gap is small either way -- see the numbers above.

moe_necessity

### 14.5 Feature-group ablation: is crime worth what it costs?

This section runs last in §14, after the model-design verdicts (14.3, 14.4) rather than
immediately after the test reveal. It isn't a property of the model just evaluated -- it's a
question about whether next iteration's data pull should look different, and answering it doesn't
require or benefit from reading the design-decision sections first.

The crime file only covers 2008–2016, and that constraint is why the *entire* modelling window
stops there — 336,079 of 418,201 available sale records (80 %) are discarded to accommodate one
borough-level feature. The rule adopted at the outset was to start with the smaller window crime
supports and widen it only if crime proved not to matter much, which makes measuring the feature's
actual contribution a prerequisite rather than an afterthought.

The method: retrain **the winning recipe** — XGBoost on the detrended target, `capped` variant,
the same configuration section 12.1 arrived at — five times, removing one feature group each time,
and compare against the full model. Using the shipped recipe matters: ablating plain CatBoost on
`log(price)` -- the target section 12.1 shows is mis-levelled -- would measure feature value on a
model already handicapped by something else.

**Scored on validation, not test.** This study's whole purpose is to *decide* something — which
features to keep, and whether the narrow window earns its cost. A decision that reads the test set
is exactly the mistake the rest of this notebook is built to avoid, so the ablation is scored on
`VAL_EVAL`. The models also early-stop on validation, which makes the absolute MdAPE here
optimistic; that is acceptable because every variant carries the identical bias and the gate below
consumes only the **delta** between variants.

**One group cannot be fully ablated.** The detrended target is `log(price / market level)`, and the
market level is built from `market_median_rolling_3m`. Removing the "market lags" group therefore
strips those columns from the model's *inputs* but cannot remove them from the *label* — the
deflator is deliberately computed from the full feature frame so the target stays identical across
all six runs. The market-lag delta below is thus a lower bound: it measures their value as
predictors only, not their total contribution to the pipeline.

**Decision rule.** If dropping crime costs less than the following gate, the 80 % data sacrifice
is not justified and the notebook's next iteration should widen the window rather than keep
squeezing more out of ten years of transactions.

**Both methodological choices above — validation scoring and the shipped recipe — change the
answer, and by enough to reverse the recommendation.** Scored on *test* and run on the *plain*
target, this same ablation reports crime as worth +0.46 pp, comfortably above the gate. Measured on
validation and on the recipe actually shipped, crime's contribution collapses to a rounding error.
The two effects push the same way: the plain-target model is so badly mis-levelled (section 12.2)
that *any* feature carrying market information looks valuable, because it is partially compensating
for the level error rather than adding signal of its own. Once detrending removes that error, most
of the apparent value goes with it — including, notably, most of the market-lag group's, whose
signal now lives in the target transform instead of in the feature matrix.

In [ ]:
ablation = ablation_study(SPLITS, VARIANTS, FEATURE_GROUPS, CONFIG, VAL_EVAL)
plot_ablation(ablation)

crime_delta = float(ablation.loc[ablation["Variant"] == "No crime", "MdAPE delta"].iloc[0])
print(f"\nCrime contributes {crime_delta:+.3f} pp of validation MdAPE.")
if crime_delta < ABLATION_GATE_PP:
    print(f"Below the {ABLATION_GATE_PP} pp gate: crime is not worth confining the model to "
          f"2008-2016. The next iteration should drop crime and widen the window to the full "
          f"1995-2024 history (~418k rows, 5x current volume).")
else:
    print(f"Above the {ABLATION_GATE_PP} pp gate: crime earns its place, and the case for "
          f"narrowing the window to keep it is real. Sourcing post-2016 LSOA crime data "
          f"(data.police.uk) would let the window widen without losing the signal.")

ablation

**The same question in the model's terms.** Below, the winning recipe is refit with six crime
designs over identical rows and an identical target — only the crime columns move. Two details
make it a test rather than a formality. Every design is refit under **five seeds**, because a
single gradient-boosted fit cannot separate a 0.1 pp effect from run-to-run variation and this
study decides whether the window survives. And each gain is paired **within** its seed before
averaging, so seed-level variation cancels rather than accumulating.

In [ ]:
CRIME_SEEDS = (42, 43, 44, 45, 46)
crime_designs = crime_resolution_study(SPLITS, VARIANTS, CONFIG, VAL_EVAL, seeds=CRIME_SEEDS)

print()
print(crime_designs.to_string(index=False, formatters={
    "MdAPE mean": "{:.3f}%".format, "MdAPE sd": "{:.3f}".format,
    "Gain vs. no crime": "{:+.3f}pp".format, "Gain sd": "{:.3f}".format}))

In [ ]:
# The same 0.15 pp bar, now applied to the best crime design available rather than to the
# only one that had ever been tried. A gain also has to be large against the seed noise it
# was measured through, or the gate is just reading a number the experiment cannot resolve.
best_crime = crime_designs.loc[crime_designs["Gain vs. no crime"].idxmax()]
noise_floor = crime_designs["MdAPE sd"].mean()

print(f"Best design      : {best_crime['Crime features']}")
print(f"Mean gain        : {best_crime['Gain vs. no crime']:+.3f} pp "
      f"(sd {best_crime['Gain sd']:.3f}, won {int(best_crime['Seeds won'])}"
      f"/{int(best_crime['Seeds'])} seeds)")
print(f"Seed noise floor : {noise_floor:.3f} pp, the mean sd of one design across seeds")
print(f"Gate             : {ABLATION_GATE_PP} pp\n")

clears = best_crime["Gain vs. no crime"] > ABLATION_GATE_PP
beats_noise = best_crime["Gain vs. no crime"] > 2 * noise_floor
if clears and beats_noise:
    print("Crime earns its place at LSOA grain, by a margin the seed spread cannot explain.\n"
          "The case for confining the model to 2008-2016 is real, and widening the window\n"
          "means sourcing post-2016 LSOA crime from data.police.uk rather than dropping it.")
elif clears:
    print("The gain clears the gate but not the seed noise it was measured through.\n"
          "Keeping 2008-2016 -- and discarding 86% of the sale records -- on a difference\n"
          "this experiment cannot resolve would be reading a decision off nothing.")
else:
    print("Crime does not clear the gate at its best resolution, as a count or a rate, whole\n"
          "or split by category. The exploratory and modelling answers agree: the window can\n"
          "widen, and section 18 records what that costs.")

### 14.6 What a property's own history is worth

The source file is a price *history*, and the pipeline has been reading it as a transaction log.
418,201 rows cover 137,760 addresses; after removing the 24.5 % that are exact duplicates,
314,895 distinct sales remain and 66 % of addresses still record two or more. **61.1 % of
in-window sales have an earlier sale of the same property somewhere in the file**, and until now
not one feature used that.

The omission is expensive in a way that is easy to check without any model at all. Take each
sale's own previous price, scale it by a crude market index, and call that the prediction: one
line of arithmetic scores **15.64 %** MdAPE against this pipeline's 13.30 %. A single column
recovers most of what twenty-four features and seventeen models achieve.

These features are causally clean — they read only sales strictly before the row being predicted
— but "strictly before" is load-bearing, and three things could break it. The 102,527 duplicate
rows are removed first, or an as-of merge can return the very row it is meant to predict.
Same-day conflicting prices are collapsed to a median. And `assert_no_lookahead` is a hard check
in section 9, with a matching assertion in the section 17 self-checks.

Measured on the same protocol as the crime study in 14.5 — same gate, same seeds, gains paired
within each seed:

In [ ]:
prior_sale_designs = prior_sale_study(SPLITS, VARIANTS, CONFIG, VAL_EVAL, seeds=(42, 43, 44))

print()
print(prior_sale_designs.to_string(index=False, formatters={
    "MdAPE mean": "{:.3f}%".format, "MdAPE sd": "{:.3f}".format,
    "Gain vs. no prior sale": "{:+.3f}pp".format, "Gain sd": "{:.3f}".format}))

In [ ]:
best_prior = prior_sale_designs.loc[prior_sale_designs["Gain vs. no prior sale"].idxmax()]
print(f"\nBest design : {best_prior['Prior-sale features']}")
print(f"Mean gain   : {best_prior['Gain vs. no prior sale']:+.3f} pp "
      f"(sd {best_prior['Gain sd']:.3f}), won {int(best_prior['Seeds won'])}"
      f"/{int(best_prior['Seeds'])} seeds")
print(f"Gate        : {ABLATION_GATE_PP} pp\n")

if best_prior["Gain vs. no prior sale"] > ABLATION_GATE_PP:
    print("Comfortably clear, and on every seed. For scale, the same protocol scores crime at\n"
          "its best resolution below +0.07 pp with a standard deviation larger than its mean.\n"
          "These four columns are therefore carried in FEATURES; the two dropped from the\n"
          "candidate set -- log_prev_sale_price and n_prior_sales -- earned nothing, since a\n"
          "tree splits on order and the log of a column it already holds is the same ordering.")
else:
    print("Below the gate: the prior-sale group does not justify its complexity.")

---
## 15. Conformal safety bound and the flip scanner

A point estimate is not enough to justify spending money. What an investor needs is a **floor**: a
value the property is very unlikely to be worth less than.

**Multiplicative split conformal prediction** supplies one. On a dedicated **calibration split**
-- separate from the validation set used for early stopping and model selection -- we compute the
ratio of actual to predicted price for every property:

$$r_i = \frac{y_i}{\hat{y}_i}$$

and take the 10th percentile, $q_{10}$. For a new property the floor is $\hat{y} \times q_{10}$,
and by construction roughly 90 % of properties should sit above it.

**Why ratios rather than differences.** Absolute residuals in this market are heteroscedastic — a
£3 M house misses by far more pounds than a £200 k flat while being no less accurate in percentage
terms. A single absolute quantile would therefore be far too loose at the bottom of the market and
far too tight at the top. The ratio form scales the buffer with the price, so one calibration
serves both tiers.

**On the exchangeability assumption.** Classical conformal prediction assumes calibration and test
data are exchangeable, which a chronological split does not strictly guarantee — the market drifts.
The empirical coverage check below is therefore not a formality; it is the actual evidence that the
guarantee holds, and it is reported honestly whether or not it lands on target.

A property is flagged as a **flip candidate** when its transaction price falls below the floor.
Calibration and scanning both call `bundle.predict`, so the model being calibrated is provably the
model being deployed -- `BEST` from section 14, chosen on validation MdAPE alone.

**Why calibration gets its own split, not the validation set.** Split conformal prediction assumes
the calibration residuals were not used to fit the model. Validation *was* used -- for early
stopping and for choosing the winning architecture -- so a model's iteration count is implicitly
tuned to minimise error on exactly those rows. Calibrating there would make `q_10` optimistically
tight. The calibration split sits strictly between validation and test in time and is untouched by
anything except this cell.

In [ ]:
# Calibrated on CALIB_EVAL, not VAL_EVAL: the model's hyperparameters (early stopping,
# architecture choice) were already tuned against validation, so computing q_10 there would
# calibrate the safety bound on data the model has effectively already seen.
Q_SAFETY = calibrate_conformal(BEST, CALIB_EVAL, SPLITS, CONFIG.conformal_alpha)
scan = scan_for_flips(BEST, TEST_EVAL, SPLITS, Q_SAFETY)
flips = scan[scan["is_flip"]].sort_values("margin", ascending=False)

coverage = (scan["actual_price"] >= scan["safe_lower_bound"]).mean() * 100
target = (1 - CONFIG.conformal_alpha) * 100
print("\n--- Empirical coverage on the held-out test set ---")
print(f"Target confidence : {target:.2f}%")
print(f"Actual coverage   : {coverage:.2f}%   ({coverage - target:+.2f} pp)")
print(f"Properties scanned: {len(scan):,}")
print(f"Flip candidates   : {len(flips):,} ({len(flips) / len(scan) * 100:.2f}%)")
if len(flips):
    print(f"Median margin     : \N{POUND SIGN}{flips['margin'].median():,.0f}")
flips.head(10)

In [ ]:
plot_flip_margins(flips, scan, Q_SAFETY, CONFIG)

---
## 16. Persisting the run

A model that exists only inside a kernel session is not a deliverable. This cell writes the
selected estimators, the conformal multiplier, the full leaderboard and a machine-readable
manifest to `artifacts/`.

Note on scope: the `predict` closures built in section 12 are not picklable by design, so what is
persisted is the underlying estimators plus the manifest needed to rebuild the closure. Loading a
bundle means running the definition cells in this notebook and calling the matching trainer's
predict logic — the notebook is the deployment unit, which is the deliberate trade-off of keeping
the whole project in a single file.

In [ ]:
persist_run(BEST, Q_SAFETY, CONFIG, RESULTS, SPLITS)

---
## 17. Automated self-checks

A notebook with no assertions is a notebook that fails silently. These checks encode the invariants
this pipeline depends on, and each one guards a defect that was actually present in the original:

1. **Chronological integrity** — no training row may be dated after a validation row.
2. **No dead feature columns** — the bug that left four categorical columns entirely NaN would
   have been caught here immediately.
3. **Categorical levels are pinned from training**, so a category code means the same thing at
   fit time and predict time.
4. **Lagged market features never see the present** — a month's own median must not equal its
   own predictor.
5. **Conformal coverage** lands near its nominal level — asserted on a *held-out slice of the
   calibration split*, never on test. The multiplier is refitted on the first 70 % of calibration
   and scored on the remaining 30 %, because checking coverage on the same rows the multiplier was
   fitted on would be an identity, not a test. Test-set coverage is printed alongside for
   information and deliberately carries no assertion: making a run fail on a held-out metric turns
   that metric into a tuning signal.
6. **Re-runnability** — the raw station table still exists and is untouched, which the original's
   variable clobbering broke.

Run them after any change to the pipeline.

In [ ]:
run_self_checks(SPLITS, CONFIG, BEST, coverage, target, RAW, df_master)

---
## 18. Limitations and where to take this next

### What this model is not

* **The scanner evaluates completed sales, not properties you can buy.** `TARGET` is
  `history_price` -- what a property *actually sold for*. A property that sold below its floor in
  2016 validates the valuation model; it is not a listing anyone can act on today. Turning this
  into a live scanner needs a listings feed (asking prices) and a calibration step for the gap
  between asking and sold price, which is a different distribution and not something this dataset
  contains.
* **The "flip candidate" flag is a statistical claim, not a financial one.** It says the price is
  below a calibrated valuation floor. It says nothing about stamp duty, refurbishment cost,
  holding cost, agent fees or the reason the property is cheap — and properties are usually cheap
  for a reason the data does not record (short lease, structural problems, a motivated seller).
  Treat the margin as a screening signal, not an expected profit.
* **Conformal coverage is marginal, not conditional.** Roughly 90 % of properties sit above the
  floor *overall*. That does not guarantee 90 % within Kensington, or within the top price decile.
* **The 2008–2016 window ends a decade ago.** Brexit, the 2016 stamp duty surcharge, the pandemic
  and the 2022 rate cycle all fall outside it. Nothing here should be pointed at today's market
  without recalibration.
* **Gain-based feature importance is not causal**, and this feature set is heavily collinear
  (latitude, longitude, borough, outcode and distance-to-centre all encode "where").
* **The evaluation universe is defined using the target.** Two filters decide which rows are
  eligible to be scored, and both read `price`: the £1,500/sqm floor (section 7, applied before the
  split, to all four slices) and the £4 M cap (section 10). Both are defensible as definitions of
  "the standard market this product serves", and neither is a temporal leak — but neither is
  identifiable at prediction time either. The reported MdAPE therefore describes a population you
  could not actually select in production, where price is the unknown.

### How this design was reached — the part a results table cannot show

Every *decision* in the finished notebook is made on training or validation data: early stopping,
model selection, the feature-group ablation (14.5), the deflator choice (14.3) and the
Mixture-of-Experts verdict (14.4). The test split is read only to report. But that describes the
code as it stands, not the path that produced it, and two design choices were reached by looking at
test-set outcomes:

* **The split ordering.** Two layouts were tried — calibration adjacent to test, versus validation
  adjacent to test — and the current one was kept because it produced smaller validation-to-test
  drift for the tree models. That comparison is only computable by scoring on test, so the
  experimental frame every other number sits inside was chosen with test-set knowledge.
* **The detrended target**, which is the project's headline result. The extrapolation bias that
  motivated it was first noticed as a +£128,318 mean residual *on test*. Section 12.2 now re-derives
  the same diagnostic on validation, which demonstrates the conclusion was reachable without the
  held-out set — but it does not change the fact that this is the order it actually happened in.

Neither can be undone without new data, and no amount of downstream discipline erases them. They
are recorded here because a notebook that claims a clean protocol while quietly having used test to
shape its own design is doing the thing this section exists to prevent. The practical consequence:
treat the headline test MdAPE as *mildly optimistic* — it is a fair estimate for the chosen
pipeline, but the pipeline itself was not chosen in complete ignorance of it.

### Ranked improvements

**1 — Split by property, not only by time.** Section 14.1 quantifies the overlap. Add a
group-aware split keyed on `fullAddress` and report both numbers; if the gap is large, the
headline metric is measuring memorisation as much as valuation.

**2 — Walk-forward backtesting.** One 60/15/10/15 cut yields one number with no error bar. Rolling
origin evaluation (expanding window, refit each year) gives a distribution of MdAPE and reveals
whether performance depends on which slice of the cycle you happened to test on.

**3 — Make the margin an actual P&L.** Add acquisition costs (stamp duty bands including the 3 %
additional-property surcharge), refurbishment, financing at the prevailing rate, agent and legal
fees. `margin` becomes expected profit, and the scanner can rank by return rather than by pounds.

**4 — Conditional (Mondrian) conformal prediction.** Calibrate separate multipliers per borough
and per price decile so coverage holds *within* the segments an investor actually shops in.

**5 — Features the data supports but the model ignores.** LSOA-level crime instead of
borough-level (a 147× resolution gain — 4,835 LSOAs against 33 boroughs — that is already in the
raw file), travel *time* to Zone 1 rather than straight-line distance, and lease length, which
drives a large share of flat valuation and is absent entirely.

**6 — Interpretability that survives collinearity.** SHAP values on the winning model, plus
permutation importance grouped over the location block, so "where" is credited once rather than
split five ways.

**7 — Quantile regression as a conformal alternative.** Fitting the 10th percentile directly
(`objective='reg:quantileerror'`) gives a floor that adapts per property, where the current
multiplicative bound applies one global ratio to everything.

**8 — Operational hardening.** *Partly done.* Sections 1–17 now live in `src/lff/`, with a
`pytest` suite over a committed 500-row fixture that runs the leakage probes in about two seconds
rather than only at the end of a full run, and `nbstripout` wired into a pre-commit hook so outputs
never land in git. Still outstanding: a retraining job that fails loudly when the section 17
self-checks do.

**9 — The dataset is a repeat-sales panel, and the model does not know it.** 418,201 rows cover
137,760 unique addresses. After removing the 24.5 % of rows that are exact duplicates,
314,895 distinct sales remain and 66.0 % of addresses still record two or more of them;
61.1 % of in-window transactions have a prior sale of the same property in the file. Not one prior-sale
feature exists in `FEATURES`. Predicting a sale as its own previous price scaled by a crude market
index — no machine learning at all — scores 15.64 % MdAPE against this pipeline's 13.30 %. A
property's own transaction history is both the strongest unused signal here and the one a flipper
actually reasons with.